# ElasticSearch - Amazon Reviews Database

In [1]:
import sys
import os
import time
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from elasticsearch import Elasticsearch, helpers

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from common.const import (
    ELASTICSEARCH_HOST, ELASTICSEARCH_USERNAME, ELASTICSEARCH_PASSWORD,
    TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS, TABLE_5CORE_TIMESTAMP_DICT, SPLIT_LIST
)
from common.helper import get_categories, load_hf_data, parse_price, parse_product_images, parse_review_images, convert_list_of_values_to_string

categories = ["All_Beauty"]  # team instruction: load All_Beauty only
BATCH_SIZE = 1000  # ES bulk insert recommended batch size

## Connect to ElasticSearch

In [2]:
def connect_elasticsearch():
    client = Elasticsearch(
        [ELASTICSEARCH_HOST],
        http_auth=(ELASTICSEARCH_USERNAME, ELASTICSEARCH_PASSWORD),
        request_timeout=600
    )
    info = client.info()
    print(f"Connected to ElasticSearch cluster: {info['cluster_name']}")
    return client

es = connect_elasticsearch()

Connected to ElasticSearch cluster: c01507f5d8134708a5efd2aa37cacb38


## Create Indices with Optimized Mappings

Key indexing decisions:
- `parent_asin`: **keyword** (exact match) : fast lookups for Q1, Q2, Q4, Q5
- `user_id`: **keyword** : fast lookups for Q4
- `title` + `description`: **text** with english analyzer : enables full-text search for Q3
- `timestamp`: **date** (epoch_millis) : enables sorting for Q2
- `rating`: **byte** : numeric aggregations for Q5

In [3]:

# Set up index names using the constants from const.py
PRODUCTS_INDEX = TABLE_RAW_PRODUCTS  # raw_products
REVIEWS_INDEX  = TABLE_RAW_REVIEWS   # raw_reviews
## Products Index Mapping:It tells ES what data type each field is and how to index it
products_mapping = {
    "settings": {
        "number_of_shards": 1,# only 1 shard since this is a small dataset
        "number_of_replicas": 0,# no replicas 
        "analysis": {
            "analyzer": {
                # custom analyzer based on the built-in "english" analyzer
                # this automatically:
                #  lowercases all text (satisfies team's Q3 requirement)
                # 2) removes stop words 
                # 3) applies stemming 
                "english_custom": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            # ID / Lookup fields :
            # "keyword" type = exact match(no tokenization)
            "parent_asin":       {"type": "keyword"},
            "product_category":  {"type": "keyword"},
            "main_category":     {"type": "keyword"},
            "store":             {"type": "keyword"},
            # Full-text search fields (used for Query 3)
            # "text" type = tokenized and analyzed for full-text search
            # using english_custom analyzer:
            # text is lowercased at index time
            # so when we lowercase the keyword at query time too,
            # both sides match 
            "title": {
                "type": "text",
                "analyzer": "english_custom",
                "fields": {"raw": {"type": "keyword"}}  #"raw" sub-field stores original value as keyword
                                                        # useful if we ever need exact match or sorting on title
            },
            "description":  {"type": "text", "analyzer": "english_custom"},
            "features":     {"type": "text", "analyzer": "english_custom"},
            # Numeric fields
            "average_rating": {"type": "float"},
            "rating_number":  {"type": "integer"},
            "price":          {"type": "double"},
            "image_count":    {"type": "integer"},
            "variant_count":  {"type": "integer"},
            # Fields we do not search on 
            # index: False so ES will store the value but NOT index it
            # this saves disk space and speeds up ingestion
            "first_image":    {"type": "keyword", "index": False},
            "videos":         {"type": "keyword", "index": False},
            "categories":     {"type": "text"},
            "details":        {"type": "object", "enabled": False},#enabled: False so ES stores it as raw JSON
                                                                 # without parsing or indexing any of its sub-fields
            "bought_together": {"type": "keyword", "index": False},
            "subtitle":       {"type": "text"},
            "author":         {"type": "keyword"}
        }
    }
}
#Review index mapping
reviews_mapping = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    #Lookup fields (used for Q2,Q4,Q5)
    "mappings": {
        "properties": {
            "parent_asin":       {"type": "keyword"},  # Q2, Q5 lookup
            "user_id":           {"type": "keyword"},  # Q4 lookup
            "asin":              {"type": "keyword"},
            "product_category":  {"type": "keyword"},
            # Timestamp as epoch millis :It allows efficient sorting for Q2 (most recent reviews)
            "timestamp":    {"type": "date", "format": "epoch_millis"},
            # Review content
            "title":        {"type": "text"},
            "text":         {"type": "text"},
            # Rating as byte (integer type),used for aggregations for Q5
            "rating":           {"type": "byte"},
            "helpful_vote":     {"type": "integer"},
            "verified_purchase": {"type": "boolean"},
            "first_image":      {"type": "keyword", "index": False},#image not searched
            "image_count":      {"type": "integer"}
        }
    }
}

# Drop and recreate indices,Create the indices
# If they already exist, drop them first to avoid conflicts
for index, mapping in [(PRODUCTS_INDEX, products_mapping), (REVIEWS_INDEX, reviews_mapping)]:
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
        print(f"Dropped index: {index}")
    es.indices.create(index=index, body=mapping)
    print(f"Created index: {index}")

Dropped index: raw_products
Created index: raw_products


C:\Users\paran\AppData\Local\Temp\ipykernel_36804\3039318855.py:98: DeprecationWarning: The 'body' parameter is deprecated for the 'create' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.indices.create(index=index, body=mapping)


Dropped index: raw_reviews
Created index: raw_reviews


## Import Data to `raw_products` index

In [4]:
def time_ingestion(fn):
    """Decorator to time ingestion"""
    start = time.time()
    fn()
    elapsed = time.time() - start
    print(f"Ingestion time: {elapsed:.2f}s")

def ingest_products():
    batch = []
    total_inserted = 0

    for category in categories:
        data = load_hf_data(root_split="raw", data_type="meta", category=category)
        for record in tqdm(data, desc=f"Products - {category}"):
            title = record['title']# clean up title , remove leading or trailing whitespace
            if title: title = title.strip()

            first_image, image_count, variant_count = parse_product_images(record['images'])
            details = json.loads(record["details"])
            details = details if details else None
            # build the ES document
            doc = {
                "_index": PRODUCTS_INDEX,
                "_id":    record["parent_asin"],  # Use parent_asin as doc ID : deduplication + fast retrieval
                "_source": {
                    "main_category":  record["main_category"],
                    "title":          title,
                    "average_rating": record["average_rating"],
                    "rating_number":  record["rating_number"],
                    "features":       convert_list_of_values_to_string(record["features"]),
                    "description":    convert_list_of_values_to_string(record["description"]),
                    "price":          parse_price(record["price"]),
                    "first_image":    first_image,
                    "image_count":    image_count,
                    "variant_count":  variant_count,
                    "videos":         json.dumps(record["videos"]),
                    "store":          record["store"],
                    "categories":     convert_list_of_values_to_string(record["categories"]),
                    "details":        details,
                    "parent_asin":    record["parent_asin"],
                    "bought_together": record["bought_together"],
                    "subtitle":       record["subtitle"],
                    "author":         record["author"],
                    "product_category": category
                }
            }
            batch.append(doc)
            # when batch reaches BATCH_SIZE, send to ES using bulk API
            # bulk API sends all documents in one HTTP request
            # chunk_size=1000 , ES sends 1000 docs per HTTP request
            # this is 2x the ES default of 500, chosen because:
            # - stays close to ES recommended default.
            # - reduces risk of 429 (rate limit) .
            if len(batch) >= BATCH_SIZE:
                helpers.bulk(es, batch,chunk_size=1000)
                total_inserted += len(batch)
                batch.clear()

        if batch:
            helpers.bulk(es, batch,chunk_size=1000)
            total_inserted += len(batch)
            batch.clear()

    print(f"Total products inserted: {total_inserted}")

time_ingestion(ingest_products)

Products - All_Beauty: 112590it [04:43, 396.86it/s] 


Total products inserted: 112590
Ingestion time: 292.15s


## Import Data to `raw_reviews` index

In [5]:
def ingest_reviews():
    batch = []
    total_inserted = 0

    for category in categories:
        data = load_hf_data(root_split="raw", data_type="review", category=category)
        for record in tqdm(data, desc=f"Reviews - {category}"):
            title = record['title']
            if title: title = title.strip()
            text = record['text']
            if text: text = text.strip()

            first_image, image_count = parse_review_images(record["images"])

            doc = {
                "_index": REVIEWS_INDEX,
                 # Composite ID: parent_asin + user_id + timestamp for uniqueness: build the ES document
                 #  same product can have many reviews (same parent_asin)
                 #  same user can review many products (same user_id)
                 #  same user could review same product  at different times
                 #  combining all 3 guarantees each review has a unique _id
                "_id": f"{record['parent_asin']}_{record['user_id']}_{record['timestamp']}",
                "_source": {
                    "timestamp":         record["timestamp"],
                    "user_id":           record["user_id"],
                    "parent_asin":       record["parent_asin"],
                    "asin":              record["asin"],
                    "rating":            record["rating"],
                    "title":             title,
                    "text":              text,
                    "first_image":       first_image,
                    "image_count":       image_count,
                    "helpful_vote":      record["helpful_vote"],
                    "verified_purchase": record["verified_purchase"],
                    "product_category":  category
                }
            }
            batch.append(doc)

            if len(batch) >= BATCH_SIZE:
                helpers.bulk(es, batch,chunk_size=1000)
                total_inserted += len(batch)
                batch.clear()

        if batch:
            helpers.bulk(es, batch,chunk_size=1000)
            total_inserted += len(batch)
            batch.clear()

    print(f"Total reviews inserted: {total_inserted}")

time_ingestion(ingest_reviews)

Reviews - All_Beauty: 701528it [14:58, 780.80it/s] 


Total reviews inserted: 701528
Ingestion time: 907.36s


## Measure Query Performance

Same test IDs as ClickHouse for fair comparison.

In [6]:
num_iter = 10 #number of runs per query 

#list for fair comparison with team
random_product_list = [
    'B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W',
    'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM'
]

random_keywords = [
    "Plumping Whip", "Eye Sticker",
    "Hair Scrunchies Colorful", "facial jade roller",
    "Professional Supreme Performance Braiding", "vintage English floral fragrance",
    "Water Weightless Hydration Conditioner",
    "hottest beauty product of 2019", "Mist Coolant Lubrication Spray System",
    "keep the skin calm and soothed"
]

random_user_list = [
    'AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA',
    'AFXCPSCWLOSVQZHN7FAECOYAT3RA', 'AHT6VS3G3JSA7KBOCBT6KPUCJX7A',
    'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA',
    'AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA',
    'AHA272AOVKWZUXL672G653A2MTIA', 'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ'
]

def measure_es_query(fn):
    """Use ES built-in took (ms) for measure query duration 
    # took is the server-side execution time in milliseconds"""
    start = time.time()
    result = fn()
    if isinstance(result, dict) and 'took' in result:
        duration_ms = result['took']  
    else:
        duration_ms = (time.time() - start) * 1000
    return duration_ms, result
def warmup(fn, runs=3):  # without warmup, the first 1-2 runs would be unfairly slow
    # and would skew the mean. (ES JVM cache :Java needs a few runs to JIT-compile the query path
    #and OS file cache : loads index segments from disk into memory)
    # we use runs=3 as it is enough to stabilise performance
    
    for _ in range(runs):
        try: fn()
        except: pass
def report(label, logs):
    print(f"\n=== {label} ===")
    print(f"Over {len(logs)} runs:")
    print(f"  Mean   : {np.mean(logs):.3f} ms")
    print(f"  Median : {np.median(logs):.3f} ms")
    print(f"  Min    : {np.min(logs):.3f} ms")
    print(f"  Max    : {np.max(logs):.3f} ms")

### Query 1: Given a product ID (parent_asin), retrieve the product title and full product description.

In [7]:
# Warmup
warmup(lambda: es.search(index=PRODUCTS_INDEX, body={
    "_source": ["title", "description"],
    "query": {"term": {"parent_asin": random_product_list[0]}}
}))

# Strategy: term query on keyword field parent_asin
# Follows same approach as ClickHouse
# Uses ES built-in took for server-side timing  

query_duration_logs = []
for item in random_product_list:
    body = {
        "_source": ["title", "description"],
        "query": {"term": {"parent_asin": item}}
    }
    duration_ms, resp = measure_es_query(lambda: es.search(index=PRODUCTS_INDEX, body=body))
    query_duration_logs.append(duration_ms)

report("Query 1: Product title & description by parent_asin", query_duration_logs)

C:\Users\paran\AppData\Local\Temp\ipykernel_36804\1083457550.py:2: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  warmup(lambda: es.search(index=PRODUCTS_INDEX, body={
C:\Users\paran\AppData\Local\Temp\ipykernel_36804\1083457550.py:17: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  duration_ms, resp = measure_es_query(lambda: es.search(index=PRODUCTS_INDEX, body=body))



=== Query 1: Product title & description by parent_asin ===
Over 10 runs:
  Mean   : 0.400 ms
  Median : 0.000 ms
  Min    : 0.000 ms
  Max    : 1.000 ms


### Query 2: Given a product ID, retrieve the most recent N customer reviews.

In [8]:


warmup(lambda: es.search(index=REVIEWS_INDEX, body={
    "size": 10, "query": {"term": {"parent_asin": random_product_list[0]}},
    "sort": [{"timestamp": {"order": "desc"}}]
}))

# Strategy: term filter on keyword field parent_asin, sort by timestamp desc.
# timestamp date field for efficient sorting.
N = 10

query_duration_logs = []
for item in random_product_list:
    body = {
        "size": N,
        "_source": ["timestamp", "title", "text", "rating"],
        "query": {
            "term": {"parent_asin": item}
        },
        "sort": [{"timestamp": {"order": "desc"}}]
    }
    duration_ms, resp = measure_es_query(lambda: es.search(index=REVIEWS_INDEX, body=body))
    query_duration_logs.append(duration_ms)

report("Query 2: Most recent N reviews by parent_asin", query_duration_logs)

C:\Users\paran\AppData\Local\Temp\ipykernel_36804\1777386997.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  warmup(lambda: es.search(index=REVIEWS_INDEX, body={



=== Query 2: Most recent N reviews by parent_asin ===
Over 10 runs:
  Mean   : 1.200 ms
  Median : 1.000 ms
  Min    : 0.000 ms
  Max    : 2.000 ms


C:\Users\paran\AppData\Local\Temp\ipykernel_36804\1777386997.py:20: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  duration_ms, resp = measure_es_query(lambda: es.search(index=REVIEWS_INDEX, body=body))


### Query 3: Given a keyword, return all products whose title or description contains it.

In [9]:

warmup(lambda: es.search(index=PRODUCTS_INDEX, body={
    "_source": ["parent_asin", "title", "description"],
    "query": {"multi_match": {"query": "beauty", "fields": ["title", "description"], "type": "phrase"}}
}))
# Strategy: multi_match full-text search across title and description
# lowercase BOTH the keyword and the field values.
# lowercase normalizer on keyword subfields and .lower() on the query string.
# Since our fields use the english analyzer (which lowercases by default during indexing),
# we only need to lowercase the query string to match.

query_duration_logs = []
for keyword in random_keywords:
    keyword_lower = keyword.lower()  
    body = {
        "_source": ["parent_asin", "title", "description"],
        "query": {
            "multi_match": {
                "query": keyword_lower,
                "fields": ["title", "description"],
                "type": "phrase"  # phrase match to stay equivalent to ClickHouse 
            }
        }
    }
    duration_ms, resp = measure_es_query(lambda: es.search(index=PRODUCTS_INDEX, body=body))
    query_duration_logs.append(duration_ms)

report("Query 3: Full-text keyword search on title & description", query_duration_logs)

C:\Users\paran\AppData\Local\Temp\ipykernel_36804\1822629970.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  warmup(lambda: es.search(index=PRODUCTS_INDEX, body={



=== Query 3: Full-text keyword search on title & description ===
Over 10 runs:
  Mean   : 1.900 ms
  Median : 2.000 ms
  Min    : 1.000 ms
  Max    : 3.000 ms


C:\Users\paran\AppData\Local\Temp\ipykernel_36804\1822629970.py:24: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  duration_ms, resp = measure_es_query(lambda: es.search(index=PRODUCTS_INDEX, body=body))


### Query 4: Given a user ID, retrieve all products reviewed by that user.

In [10]:

warmup(lambda: es.search(index=REVIEWS_INDEX, body={
    "_source": ["user_id", "parent_asin", "rating", "timestamp"],
    "query": {"term": {"user_id": random_user_list[0]}}
}))
# Strategy: term filter on keyword field user_id.
# keyword mapping : exact match, no tokenization 

query_duration_logs = []
for user in random_user_list:
    body = {
        "_source": ["user_id", "parent_asin", "rating", "timestamp"],
        "query": {
            "term": {"user_id": user}
        }
    }
    duration_ms, resp = measure_es_query(lambda: es.search(index=REVIEWS_INDEX, body=body))
    query_duration_logs.append(duration_ms)

report("Query 4: All reviews by user_id", query_duration_logs)

C:\Users\paran\AppData\Local\Temp\ipykernel_36804\4046608465.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  warmup(lambda: es.search(index=REVIEWS_INDEX, body={
C:\Users\paran\AppData\Local\Temp\ipykernel_36804\4046608465.py:16: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  duration_ms, resp = measure_es_query(lambda: es.search(index=REVIEWS_INDEX, body=body))



=== Query 4: All reviews by user_id ===
Over 10 runs:
  Mean   : 0.600 ms
  Median : 1.000 ms
  Min    : 0.000 ms
  Max    : 1.000 ms


### Query 5: Retrieve average rating, rating distribution, and total reviews for a product.

In [11]:

warmup(lambda: es.search(index=REVIEWS_INDEX, body={
    "size": 0,
    "query": {"term": {"parent_asin": random_product_list[0]}},
    "aggs": {"avg_rating": {"avg": {"field": "rating"}}}
}))
# Strategy: term filter + aggregations.
# team instruction: explicitly retrieve avg_rating, total_reviews,
# and individual counts for each star level (1–5) — one query per product.

query_duration_logs = []
for item in random_product_list:
    body = {
        "size": 0,  # aggregations only
        "query": {
            "term": {"parent_asin": item}
        },
        "aggs": {
            "avg_rating":      {"avg":         {"field": "rating"}},
            "total_reviews":   {"value_count": {"field": "rating"}},
            "one_star_reviews":   {"filter": {"term": {"rating": 1}}},
            "two_star_reviews":   {"filter": {"term": {"rating": 2}}},
            "three_star_reviews": {"filter": {"term": {"rating": 3}}},
            "four_star_reviews":  {"filter": {"term": {"rating": 4}}},
            "five_star_reviews":  {"filter": {"term": {"rating": 5}}}
        }
    }
    duration_ms, resp = measure_es_query(lambda: es.search(index=REVIEWS_INDEX, body=body))
    query_duration_logs.append(duration_ms)

report("Query 5: Rating stats & per-star counts by parent_asin", query_duration_logs)

C:\Users\paran\AppData\Local\Temp\ipykernel_36804\4155934151.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  warmup(lambda: es.search(index=REVIEWS_INDEX, body={
C:\Users\paran\AppData\Local\Temp\ipykernel_36804\4155934151.py:27: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  duration_ms, resp = measure_es_query(lambda: es.search(index=REVIEWS_INDEX, body=body))



=== Query 5: Rating stats & per-star counts by parent_asin ===
Over 10 runs:
  Mean   : 4.500 ms
  Median : 4.000 ms
  Min    : 3.000 ms
  Max    : 10.000 ms


## Load 5core Timestamp Data (train/valid/test splits)

In [12]:
for split, table in TABLE_5CORE_TIMESTAMP_DICT.items():
    if es.indices.exists(index=table):
        es.indices.delete(index=table)
    es.indices.create(index=table, body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {"properties": {
            "user_id":     {"type": "keyword"},
            "parent_asin": {"type": "keyword"},
            "rating":      {"type": "byte"},
            "timestamp":   {"type": "date", "format": "epoch_millis"}
        }}
    })
    print(f"Created index: {table}")
    batch = []
    total = 0
    for category in categories:
        data = load_hf_data(root_split="5core", data_type="timestamp", split=split, category=category)
        for record in tqdm(data, desc=f"{split} - {category}"):
            doc = {
                "_index": table,
                "_id": f"{record['user_id']}_{record['parent_asin']}_{record['timestamp']}",
                "_source": {
                    "user_id":     record["user_id"],
                    "parent_asin": record["parent_asin"],
                    "rating":      int(float(record["rating"])),
                    "timestamp":   record["timestamp"]
                }
            }
            batch.append(doc)
            if len(batch) >= BATCH_SIZE:
                helpers.bulk(es, batch)
                total += len(batch)
                batch.clear()
        if batch:
            helpers.bulk(es, batch,chunk_size=1000)
            total += len(batch)
            batch.clear()
    print(f"Inserted {total} records into {table}")

C:\Users\paran\AppData\Local\Temp\ipykernel_36804\795513524.py:4: DeprecationWarning: The 'body' parameter is deprecated for the 'create' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.indices.create(index=table, body={


Created index: 5core_timestamp_train


train - All_Beauty: 2237it [00:03, 742.83it/s]


Inserted 2237 records into 5core_timestamp_train
Created index: 5core_timestamp_valid


valid - All_Beauty: 276it [00:01, 159.07it/s]


Inserted 276 records into 5core_timestamp_valid
Created index: 5core_timestamp_test


test - All_Beauty: 22it [00:00, 25.03it/s]

Inserted 22 records into 5core_timestamp_test
